In [ ]:
import os 
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# seting global dir
cwd=Path.cwd()
if cwd.name == "reports parts":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent) 
os.getcwd()

'/home/maxi7524/repositories/final/notebooks'

# Deep Learning for Regulatory DNA Sequence Optimization: Enhancer Restoration and Synthetic Promoter Engineering

***

## Introduction
#TODO to można trochę przepisać, jest za mało raportowe a za bardzo githuboe, brakuje jeszcze wstępu o tym dlaczego ten problem jest ważny, że dzieki temu można zwiększyć prouukcje w hodowlach itd. 
This report documents the computational engineering pipeline designed to restore disrupted enhancer sequences and optimize synthetic promoter activity. Using deep learning models as an *in silico* simulation environment, we navigate the complex cis-regulatory grammar to maximize or recover the `rna_dna_ratio`. 

To achieve this, we have developed a modular, structured library architecture that enables independent manipulation of individual components, facilitating high-throughput testing across diverse models.

### Report Structure
To ensure efficient navigation across the computational experiments, this report is partitioned into the following thematic modules:
1. **Introduction**: Contextual scope, structural index, and an architectural overview of the regulatory optimization pipeline.
2. **Methodology**: Definition of hard biological constraints (`SequenceValidator`), structural formulation of the 4-stage closed-loop execution workflow, and tracking metrics.
3. **Analysis - Part I: Validation Model Selection**: Cross-model evaluation strategy (`Experiment I.1`) utilizing rolling variance-penalized trajectory weighting ($W_k(t)$).
4. **Analysis - Part II: Benchmarking Optimizers and Interpreters**: Evaluation of combinatorial search algorithms versus Beam Collapse (`Analysis II.1`), and the diagnostic profiling of Interpreter Trapping via decoupled phenotype-genotype tracking ($\Delta S(t)$, $R_M(t)$, $TI(t)$) (`Analysis II.2`).
5. **Analysis - Part III: Biological Validation**: Post-optimization compositional stability analysis (global and sliding-window GC trajectories) and functional TFBS motif restoration using Position-Specific Scoring Matrices (`Experiment III.1`).
6. **Discussion & Future Extensions**: Critical assessment of model-induced sequence biases and the mathematical formulation of a soft GC content regularization framework.

## Methodology

### Experimental design 

#### Model description
For our experiments we used three models. All of them are constructed as convolution neural networks with heads to predict latent space

| Model | Warstwy | Filtry | Kernels | Padding | Pooling |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Deepstar** | Conv1, Conv2, Conv3 | 128, 256, 256 | 7, 7, 5 | k//2 | MaxPool1d(2) |
| **Second Deepstarr** | Conv1, Conv2, Conv3, Conv4 | 246, 60, 60, 120 | 7, 3, 5, 3 | same | MaxPool1d(2) |
| **Model Original** | Conv1, Conv2 | 64, 128 | 15, 10 | 7, 5 | MaxPool1d(4), AdaptiveAvgPool1d(8) |

#### Optimizers description

#TODO - opis optimizeró (z docsów to wziąć)

#### Interpreters descriptions 

#TODO - opis 

#### Biological constraints
The practical utility of an *in silico* engineered regulatory element depends strictly on its structural and biological feasibility. Unconstrained maximization of a neural network score often yields pathological sequences that exploit model bugs (e.g., generating unnatural, extreme homopolymer repetitions or highly skewed GC-content distributions) that are non-functional or un-clonable *in vivo*.

To guarantee the reliability of the optimized enhancers, every candidate sequence is processed through a cascading short-circuit validation pipeline (`SequenceValidator`). We track and report the following sequence integrity metrics:
1. **GC-Content Equilibrium**: Tracking the rolling GC-percentage across a sliding window to ensure it stays within the physiologically viable range:
$$\text{GC}\% = \frac{N_G + N_C}{L} \in [0.25, 0.65]$$
1. **Homopolymer Constraint Compliance**: Verifying that consecutive single-nucleotide repeats do not exceed strict structural thresholds ($A/T \le 10$ consecutive bases, $G/C \le 7$ consecutive bases) to be able to synthesis them in lab.
2. **Regulatory Motif Restoration**: Evaluating whether the analytical mutations guided by the interpreters successfully reconstructed established biological consensus motifs (e.g., TATA-box, CAAT-box, or specific TFBS documented in databases like JASPAR). This verifies that the optimization process is recovering real biological grammar rather than generating adversarial sequence artifacts.


#### Workflow review

The execution process of the DNA sequence optimization pipeline consists of four chronological stages:

1. **Data Identification and Sequence Blocking:**
Classification of the dataset (`DNADataset` vs `DNADatasetNoAdapters`). If adapters are detected, the system determines their boundary coordinates (`prefix_len` and `suffix_len`), completely blocking these regions from point mutations.


2. **Environment Initialization:**
Setting the task mode (`optimization` or `constrained_recovery`) and the iteration limit. Loading the weights of the predictive neural network models, configuring biological filtration rules, and assigning interpretation and optimization algorithms.

3. **Processing Orchestration:**
Launching parallel optimization for individual data packets, where optimization algorithms cooperate directly with interpreters.

4. **Iterative Optimization and Validation Loop:**
The optimization loop of each sequence within a single iteration, encompassing four substeps:

* **Attribution:** Determination of the $L \times 4$ feature importance matrix using `Saliency`, `IntegratedGradients`, or `InSilicoMutagenesis` methods.

* **Variant Generation:** Proposal of single nucleotide substitutions based on the calculated weights (bypassing the previously blocked adapter regions).

* **Cascaded String Validation:** Immediate rejection of sequences with incorrect lengths or out-of-bounds global GC content ([0.25, 0.80]), followed by regex-based filtration of homopolymer repeats exceeding 8 for $A/T$ and 12 for $G/C$.

* **Evaluation and State Update:** Prediction of the average expression metric `rna_dna_ratio` by the neural network and updating the population using a sequential (`BeamSearch`) or stochastic (`SimulatedAnnealing`) selection method.


### Analysis Methodology

Each analytical module pairs an in silico computational experiment with a corresponding programmatic validation suite to evaluate sequence mutations systematically.

## Analysis

***

### Part I Validation Model Selection




***

#### Experiment I.1

##### Experiment design
To protect the optimization track against overfitting to a single, pathological model architecture, we implement a cross-model evaluation strategy. Let $\mathcal{M} = \{f_1, f_2, \dots, f_K\}$ represent the initial pool of available trained networks. For a given target sequence $X$, an optimizer runs an isolated optimization track guided exclusively by the target model $f_k \in \mathcal{M}$, producing a sequence mutation trajectory $\mathcal{T}_k = (X_k^{(0)}, X_k^{(1)}, \dots, X_k^{(T)})$, where $T$ is the maximum iteration depth.

To select the final validation group, every sequence snapshot $X_k^{(t)}$ within the trajectory $\mathcal{T}_k$ is concurrently evaluated by all other models in the pool:
$$S_{k, m}(t) = f_m\left(X_k^{(t)}\right) \quad \forall f_m \in \mathcal{M}$$

Instead of relying on unmeasurable statistical significance within a closed system, we calculate the rolling cross-model variance $\sigma_k^2(t)$ at each iteration step:
$$\sigma_k^2(t) = \frac{1}{K-1} \sum_{m=1}^K \left(S_{k,m}(t) - \bar{S}_k(t)\right)^2$$

To dynamically scale the reliability of each optimization path, we introduce a variance-penalized weighting function $W_k(t)$:
$$W_k(t) = \frac{1}{1 + \sigma_k^2(t)}$$

Under this formulation, when the cross-model variance is minimal ($\sigma_k^2(t) \to 0$), the weight approaches unity ($W_k(t) \sim 1$), signaling consensus across the ensemble space. As the models diverge and the variance grows, the denominator grows as $1 + \sigma_k^2(t)$, driving $W_k(t)$ toward zero. Models displaying extreme trajectory divergence (high variance causing a collapse in $W_k(t)$) are structurally discarded from the final ground truth validation group.

* **X-Axis Projection**: The iteration depth $t \in \{0, \dots, T\}$, representing the evolutionary timeline of the sequence under the guidance of model $f_k$.
* **Y-Axis Projection**: The cross-evaluated fitness scores $S_{k, m}(t)$ labeled cleanly by the validation model identity to visualize trajectory splits.

***

##### Results 


***
#### plots

In [ ]:
# CODE CHUNK: Model Trajectory Extraction and Statistical Screening
## Initialize trajectory data structures and cross-evaluation matrices
### Loop over available models to isolate performance distributions across iterations
# TODO: Implement multi-model scoring loop across saved optimization history files
# TODO: Compute rolling cross-model variance and calculate the penalized weight W_k(t)
# TODO: Generate line plots mapping Iteration vs. Individual Model Score Evaluations labeled by target model
# TODO: Filter and reject pathological networks whose variance penalty collapses the track weight below threshold

***
### Part II - Benchmarking Optimizers and Interpreters


***

#### Analysis II.1  Sequence Convergence and Structural Divergence across Optimization Layouts

##### Experimental design
When applying combinatorial search strategies (Deterministic Beam Search, Stochastic Beam Search with Metropolis-Hastings, and Boltzmann-sampled Search) to discrete nucleotide strings, we monitor the relationship between fitness maximization and sequence convergence. Given a fixed budget of allowed modifications ($M \in \{40, 16, 20\}$), multiple distinct optimization paths can yield structurally diverse sequence wainscots that map to identical localized fitness peaks.

To quantify this, we measure the pair-wise Hamming distance between the final optimized outputs generated across different initial random seeds or optimization parameters:
$$D_H(X_A, X_B) = \frac{1}{L} \sum_{i=1}^L \mathbb{I}\left(X_{A,i} \neq X_{B,i}\right)$$

This metric reveals whether our algorithms are experiencing **Beam Collapse** (where the population locks onto a single sequence topology early in the timeline) or if stochastic optimization successfully preserves structural diversity, mapping multiple unique paths across the regulatory fitness landscape.

***

##### Results

***

##### Plots



In [ ]:
# Analysis of population diversity and premature convergence tracking
## Define pairwise sequence divergence calculation suite

def compute_population_hamming_variance(sequences):
    #TODO - możesz też to ściągnać pewnie z jakieś  biblioteki
    # Compute distances between all pairs within the tracking pool
    ## Convert sequence arrays to discrete distance matrices
    n_seqs = len(sequences)
    if n_seqs <= 1:
        return 0.0
        
    distances = []
    for i in range(n_seqs):
        for j in range(i + 1, n_seqs):
            ### Calculate fraction of mismatching nucleotides
            mismatches = sum(1 for a, b in zip(sequences[i], sequences[j]) if a != b)
            distances.append(mismatches / len(sequences[i]))
            
    ## Return the average population diversity metric
    return float(np.mean(distances))

# Trajectory iteration parsing
## Extract tracking logs across historical optimization checkpoints
# TODO: Load results JSON files from target optimization runs
# TODO: Compute the pairwise Hamming variance score for each step of the beam history
# TODO: Plot Iteration vs. Mean Hamming Distance to evaluate diversity preservation (grupujemy po optimizerach i modelach żeby to zobazcyć więc kilka plotó tutaj będzie)

***

#### Analysis II.2 Interpreter Saturation and Transcription Factor Trapping



##### Experimental design

A major hazard during sequence optimization is the phenomenon of **Interpreter Trapping**. When a sequence accumulates mutations that create or optimize a highly dominant, high-affinity transcription factor binding site (TFBS), the model's localized output score can saturate.

Mathematically, if the local sequence region enters a flat plateau of the model’s activation space, the first-order instantaneous gradient of the model output $f(X)$ with respect to the input matrix $X$ drops to zero:


$$\nabla_X f(X) \approx \mathbf{0}$$

Consequently, local sensitivity methods like `SaliencyInterpreter` fail to provide meaningful signals for the remaining positions in the sequence. The optimizer becomes trapped, making random, unguided changes to adjacent bases because the model's sensitivity landscape has flattened out. We evaluate how advanced modules, such as `IntegratedGradientsInterpreter` (which evaluates integrated paths to bypass saturation) and `InSilicoMutagenesis` (which directly samples alternate states), overcome this trapping effect to maintain productive optimization.

To empirically evaluate this phenomenon under constrained logging conditions, we track the coupling between phenotypic score updates and genotypic mutation rates. We generate comparative trajectory plots mapping the mathematical collapse of score velocity alongside continuous trapping markers to visually isolate saturation zones without explicit gradient extraction.

##### Methodology: Decoupled Phenotype-Genotype Tracking

Given the absence of direct access to the gradient matrices, the detection of **Interpreter Trapping** and activation layer saturation is realized by analyzing the coupling between the sequence space (genotype) and the model's prediction space (phenotype).

The methodology assumes that within a saturated, flat region of the state-space, the directional attribution of the interpreter degrades into stochastic noise. Consequently, the optimizer shifts from gradient-guided optimization to a random walk. This state is isolated by continuously monitoring three time-correlated parameters derived directly from the logged trajectories:


1. **Phenotypic Score Velocity ($\Delta S(t)$):**

Measures the absolute fitness change between consecutive iterations:

$$\Delta S(t) = |S(t) - S(t-1)|$$

Values of $\Delta S(t) \to 0$ signal that the trajectory has entered an optimization plateau.

2. **Genotypic Mutation Rate ($R_M(t)$):**

Quantifies structural sequence modifications using the normalized Hamming distance:
$$R_M(t) = D_H\left(X^{(t)}, X^{(t-1)}\right) = \frac{1}{L} \sum_{i=1}^L \mathbb{I}\left(X^{(t)}[i] \neq X^{(t-1)}[i]\right)$$

A sustained $R_M(t) > 0$ paired with a decaying $\Delta S(t)$ proves that the optimizer is actively generating variants, but they yield no directional score advantage.

3. **Trapping Index ($TI(t)$):**

Aggregates these metrics to capture the decoupling of structural variance from functional gain:

$$TI(t) = \frac{R_M(t)}{1 + \Delta S(t)}$$

An abrupt spike in $TI(t)$ mathematically indicates that the interpreter has lost sensitivity and is feeding noise to the optimization engine, causing structural drift without phenotypic progress.

---



##### Results

---

##### Plots



In [ ]:
# Empirical tracking of phenotype-genotype decoupling
## Extract raw sequence and score data from optimization logs
# TODO: Load sequence arrays X(t) and expression scores S(t) per iteration step from the JSON run logs
# TODO: Compute the phenotypic score velocity Delta S(t) and the genotypic stepwise Hamming distance R_M(t)

## Saturated plateau and trapping evaluation
# TODO: Calculate the continuous Trapping Index TI(t) across execution timelines to isolate stagnation zones
# TODO: Generate line charts mapping Iteration vs. TI(t) stratified by the acting interpreter module to compare trapping vulnerability

***
### Part III - Biological Validation and Motif Restoration

***

#### Experiment III.1

##### Experiment design

To validate the biological viability and sequence integrity of the generated enhancers, we execute a post-optimization validation protocol focused on compositional stability and functional element restoration. Let $\Omega$ represent the set of distinct optimization algorithms under evaluation. For an initial sequence $X^{(0)} \in \mathcal{A}^L$ defined over the nucleotide alphabet $\mathcal{A} = \{A, C, G, T\}$ of length $L$, an optimizer $\omega \in \Omega$ generates a discrete state-space trajectory $\mathcal{T}_\omega = (X_\omega^{(0)}, X_\omega^{(1)}, \dots, X_\omega^{(T)})$, where $T$ denotes the maximum iteration depth.

To monitor compositional drift across the evolutionary timeline, we define the global GC content operator $f_{GC}: \mathcal{A}^L \to [0, 1]$ using the indicator function $\mathbb{I}$:


$$f_{GC}(X) = \frac{1}{L} \sum_{i=1}^L \mathbb{I}\left(X[i] \in \{G, C\}\right)$$

The continuous compositional transformation for each optimization track is tracked via the discrete set $\mathcal{G}_\omega = \{f_{GC}(X_\omega^{(t)}) \mid t \in \{0, \dots, T\}\}$. To detect localized nucleotide cluster anomalies in the final optimized variants $X_\omega^{(T)}$, we compute a spatial sliding-window profile. For a fixed window size $w \in \mathbb{N}$ where $w < L$, the localized GC density at each starting coordinate $j \in \{1, \dots, L - w + 1\}$ is formulated as:


$$\phi_{GC}(X, j, w) = \frac{1}{w} \sum_{i=j}^{j+w-1} \mathbb{I}\left(X[i] \in \{G, C\}\right)$$

Functional restoration of Transcription Factor Binding Sites (TFBS) is quantified using log-odds Position-Specific Scoring Matrices (PSSMs) obtained from the JASPAR database. Let $M$ denote a target motif matrix of width $W_M$. The raw binding affinity score $S(X, p, M)$ at position $p \in \{1, \dots, L - W_M + 1\}$ is modeled as:


$$S(X, p, M) = \sum_{k=1}^{W_M} \log_2 \left( \frac{P_M(X[p+k-1], k)}{B(X[p+k-1])} \right)$$

where $P_M(b, k)$ represents the probability of observing base $b \in \mathcal{A}$ at position $k$ within the motif model, and $B(b)$ is the background nucleotide frequency. A localized sub-sequence is classified as a restored functional motif if $S(X, p, M) \ge \theta_M$, where $\theta_M$ is a strict relative scoring threshold derived from the matrix profile. Sequence viability requires the compliance vector $\mathbb{V}(X) = 1$, indicating absolute satisfaction of homopolymer length limits ($A/T \le 8$, $G/C \le 12$) and global boundary parameters.

* **X-Axis Projection**: The iteration depth $t \in \{0, \dots, T\}$ for global trajectories, and the nucleotide coordinate index $j$ for localized spatial window analysis.
* **Y-Axis Projection**: The global/local GC ratios and the continuous log-odds affinity scores $S(X, p, M)$ evaluated across the sequence space.

---

##### Results

---

#### Plots

In [7]:
# CODE CHUNK: Biological Validation Suite, GC Drift over Time, and BioPython Motif Mapping
## Run final output sequences through the verification pipeline and document motif emergence via Bio.motifs
### Filter sequences using compliance checks and evaluate biological properties
# TODO: Extract and plot the GC content trajectory over time for each optimizer to observe compositional drift
# TODO: Generate clean distribution plots of sliding-window GC content for the optimized enhancers
# TODO: Export optimized sequences to FASTA and use Bio.motifs to detect restored TFBS from JASPAR matrices
# TODO: Compile the final performance summary table containing sequence IDs, scores, and validation flags with selected best sequenes

## Discussion & Future Extensions

### Implementation of Soft GC Bias Penalization
During our experimental evaluation, we observed that several deep learning models exhibit an unbiological bias toward maximizing global GC ratios to pump output expression predictions, pushing sequences to the upper bounds of the `SequenceValidator` filter. 

To address this in future iterations of the optimization code, we propose moving away from hard-threshold pruning toward a soft structural penalization term added directly to the optimizer's objective function. Let $f(X)$ be the raw ensemble fitness score, and $\text{GC}(X)$ be the global GC fraction of sequence $X$. The regularized objective function $\tilde{f}(X)$ is defined as:
$$\tilde{f}(X) = f(X) - \lambda \cdot \left(\text{GC}(X) - \text{GC}_{\text{WT}}\right)^2$$

where $\text{GC}_{\text{WT}}$ represents the native baseline GC content of the wild-type enhancer sequence, and $\lambda$ is a scalar regularization hyperparameter. This soft penalty forces the optimization trajectories to stay rooted within a natural biological sequence background, preventing the generation of artificial high-GC structures while preserving exploration capacity.